# Read the state list produced by Maxitor

Load the exported graph into Maxitor and request its lifecycle diagram data.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
import asyncio
import json

from pydantic import Field

from aoa.action_machine.context import Context
from aoa.action_machine.domain import BaseCoreEntity, BaseDomain, BaseStateEntity, Lifecycle, value
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from maxitor.front.model.diagrams.actions.get_lifecycle_finite_automaton_action import GetLifecycleFiniteAutomatonAction
from maxitor.front.model.diagrams.resources.duckdb_graph_resource import DUCKDB_GRAPH_CONNECTION_KEY, DuckDBGraphResource


class OrderStateEntity(BaseStateEntity):
    """Carry presentation fields and a separate domain field."""

    sort_order: int = Field(description="Order within the state list")
    color: str = Field(default="#1565c0", description="State outline colour")
    fill_color: str = Field(default="#e3f2fd", description="State fill colour")
    requires_receipt: bool = Field(default=False, description="Whether a receipt is needed")

    DRAFT = value(code="draft", display_name="Draft", sort_order=99)
    PAID = value(code="paid", display_name="Payment received", sort_order=1, requires_receipt=True)
    CANCELLED = value(code="cancelled", display_name="Cancelled", sort_order=2)


class OrderLifecycle(Lifecycle[OrderStateEntity]):
    """Keep the initial state first despite its high sort order."""

    _template = (
        Lifecycle()
        .state(OrderStateEntity.DRAFT)
        .to(OrderStateEntity.PAID, OrderStateEntity.CANCELLED)
        .initial()
        .state(OrderStateEntity.PAID)
        .final()
        .state(OrderStateEntity.CANCELLED)
        .final()
    )


class ShopDomain(BaseDomain):
    """Group the entities in this example."""

    name = "shop"
    description = "Small shop"


@entity(description="Customer order", domain=ShopDomain)
class OrderCoreEntity(BaseCoreEntity):
    """Attach the lifecycle to a model entity."""

    id: str = Field(description="Order identifier")
    lifecycle: OrderLifecycle = Field(description="Current order state")

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
async def read_states():
    """Request the real Maxitor action without starting an HTTP server."""
    machine = ActionProductMachine(loggers=[])
    graph = json.loads(machine.graph_coordinator.to_json())
    lifecycle = next(node for node in graph["nodes"] if node["type"] == "Lifecycle")
    resource = DuckDBGraphResource.build_from_json(graph)
    result = await machine.run(
        Context(),
        GetLifecycleFiniteAutomatonAction(),
        GetLifecycleFiniteAutomatonAction.Params(lifecycle_graph_node_id=lifecycle["id"]),
        {DUCKDB_GRAPH_CONNECTION_KEY: resource},
    )
    states = result.lifecycle_finite_automaton["states"]
    assert [state["key"] for state in states] == ["draft", "paid", "cancelled"]
    paid = next(state for state in states if state["key"] == "paid")
    assert "requires_receipt" not in paid["attributes"]
    assert paid["attributes"]["color"] == "#1565c0"
    print("state order:", [state["key"] for state in states])
    print("paid label:", paid["display_name"])
    print("diagram attributes:", paid["attributes"])


asyncio.run(read_states())

state order: ['draft', 'paid', 'cancelled']
paid label: Payment received
diagram attributes: {'code': 'paid', 'color': '#1565c0', 'display_name': 'Payment received', 'fill_color': '#e3f2fd', 'sort_order': 1}


## What the result means

Maxitor reads the saved graph through its real action. Initial states stay first, then presentation order controls the remaining states. The diagram keeps supported visual attributes such as colour; the business field `requires_receipt` remains in the full graph instead of appearing in this diagram response.